# Lab 1: From a vague question to a defensible description

**CS203 Data Science Fundamentals | Week 1 | Student copy**  
**Duration:** 2 hours | **Submit:** one executed notebook

Use only Week 1 ideas: a row has a unit, columns have meanings and measurement types, a sample is not automatically the population, and a pattern does not prove a cause. You will work with **one student-level table only**. We study joins, missingness, and complex data cleaning later.

The synthetic data support careful descriptive claims about the supplied sample. They do not support decisions about real students.

## Learning goals and route

- Turn a vague request into a question the available table can answer.
- Identify the unit, variables, sample, and population of interest.
- Defend an operational definition.
- Produce one descriptive table and one honest chart.
- Write a cautious conclusion with a limitation.

| Time | Activity | Output |
| --- | --- | --- |
| 0:00-0:10 | Enter ID and inspect brief | Fingerprint and case |
| 0:10-0:25 | Read data contract | Unit and variable notes |
| 0:25-0:40 | Frame a precise question | Question contract |
| 0:40-1:10 | Build descriptive evidence | evidence_table |
| 1:10-1:30 | Make and interpret chart | Figure and caption |
| 1:30-1:45 | Challenge sample or proxy | Limitation |
| 1:45-2:00 | Peer check and memo | Final notebook |

There are **10 code cells**. Six are supplied and four are yours. Your written reasoning matters as much as code.

## Before you start

1. Open the notebook in Google Colab.
2. Run the next cell to load the table.
3. In the Student ID cell, replace YOUR_OFFICIAL_STUDENT_ID with your HERO ID. Do not enter your name.
4. Run cells in order. Your ID selects an individual brief and sample.

If you see a Student ID error, replace the placeholder and run the cell again. Do not change the case-selection code.

In [ ]:
from pathlib import Path
import hashlib
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW_DATA = "https://raw.githubusercontent.com/Sabokrou/Sabokrou.github.io/main/teaching/data-science-fundamentals/datasets/students.csv"
try:
    students = pd.read_csv(Path("data") / "students.csv")
    print("Loaded local students.csv")
except FileNotFoundError:
    students = pd.read_csv(RAW_DATA)
    print("Loaded the released Week 1 student table")

print(f"Rows: {len(students):,} | Columns: {students.shape[1]}")
students.head()

In [ ]:
# Replace only the text inside quotation marks with your official HERO student ID.
#@title Enter your official student ID, then run this cell
STUDENT_CODE = "YOUR_OFFICIAL_STUDENT_ID" #@param {type:"string"}

if STUDENT_CODE.strip() in {"", "YOUR_OFFICIAL_STUDENT_ID", "REPLACE_WITH_YOUR_STUDENT_ID", "7878787878"}:
    raise ValueError("SETUP STEP: enter your official HERO ID, for example STUDENT_CODE = '12345678', then run this cell again.")
if not re.fullmatch(r"[A-Za-z0-9_-]{4,32}", STUDENT_CODE):
    raise ValueError("Use your official ID: 4-32 letters, digits, underscores, or hyphens. Do not enter your name.")

CASE_CARDS = [
    {"code": "A", "title": "Study-mode composition",
     "brief": "The Faculty says: 'Most students study in the same way, so one support schedule should fit everyone.'",
     "focus": "study_mode",
     "decision": "Describe the study-mode composition of the supplied sample and what it can, and cannot, tell the Faculty about support scheduling.",
     "caution": "A category share does not tell you why students chose that mode or whether one schedule works for everyone."},
    {"code": "B", "title": "Programme composition",
     "brief": "A colleague says: 'Computer Science is clearly the normal student profile here.'",
     "focus": "programme",
     "decision": "Describe programme composition and assess whether the phrase 'normal student profile' is supported.",
     "caution": "The largest category is not a definition of normality and does not describe every student in it."},
    {"code": "C", "title": "Entry-score comparison",
     "brief": "An adviser says: 'Part-time students arrive less prepared, so they will need different teaching.'",
     "focus": "study_mode and entry_score",
     "decision": "Describe entry-score patterns by study mode and identify a cautious next question for support planning.",
     "caution": "Entry score is only one measurement. A group difference does not explain ability, need, or what teaching causes better outcomes."},
    {"code": "D", "title": "Age as a proxy",
     "brief": "A timetable planner says: 'Older students probably need evening classes.'",
     "focus": "age_at_entry and study_mode",
     "decision": "Describe age, including any pattern by study mode, and evaluate whether age is a valid proxy for scheduling needs.",
     "caution": "Age is observable; availability, work, caring duties, and travel constraints are not."},
    {"code": "E", "title": "First-generation representation",
     "brief": "A committee asks: 'Are first-generation students visible in this student data?'",
     "focus": "first_generation",
     "decision": "Describe the first-generation share and state what else is needed before judging representation or access.",
     "caution": "A share in a dataset cannot show whether it matches the university population or whether support is equally accessible."},
    {"code": "F", "title": "Programme and entry score",
     "brief": "A manager says: 'One programme admits better-prepared students.'",
     "focus": "programme and entry_score",
     "decision": "Describe entry-score summaries by programme and decide whether the wording 'better prepared' is defensible.",
     "caution": "An entry score is an operational measure, not a complete measure of preparation, potential, or programme quality."},
]

seed = int.from_bytes(hashlib.blake2b(STUDENT_CODE.encode("utf-8"), digest_size=8).digest(), "big")
rng = np.random.default_rng(seed)
sample_size = 160 + int(seed % 61)
assigned_ids = np.sort(rng.choice(students["student_id"].to_numpy(), size=sample_size, replace=False))
case_card = CASE_CARDS[seed % len(CASE_CARDS)]
checksum = hashlib.blake2b(",".join(map(str, assigned_ids)).encode("utf-8"), digest_size=3).hexdigest().upper()
ASSIGNMENT_FINGERPRINT = f"{case_card['code']}-{sample_size}-{checksum}"

print("YOUR WEEK 1 ASSIGNMENT")
print("Fingerprint:", ASSIGNMENT_FINGERPRINT)
print("Case:", case_card["title"])
print("Vague brief:", case_card["brief"])
print("Focus:", case_card["focus"])
print("Decision:", case_card["decision"])
print("Caution:", case_card["caution"])

In [ ]:
students_sample = students.loc[students["student_id"].isin(assigned_ids)].copy()
assert len(students_sample) == sample_size
assert students_sample["student_id"].is_unique

print("One row represents one student in the supplied sample.")
print("Student IDs are unique:", students_sample["student_id"].is_unique)
students_sample.head()

In [ ]:
data_contract = pd.DataFrame([
    ["student_id", "identifier", "one student", "Do not calculate a mean or use it as a measurement."],
    ["programme", "nominal category", "declared programme", "Categories have no numerical order."],
    ["study_mode", "nominal category", "full-time or part-time", "This is not a measure of effort or need."],
    ["entry_score", "continuous number", "recorded entry score", "It is an operational measure, not complete preparation."],
    ["age_at_entry", "discrete number", "age on entry", "It may be a weak proxy for scheduling needs."],
    ["first_generation", "binary category", "first-generation status", "Use only for group-level description."],
], columns=["column", "measurement type", "meaning", "one caution"])
data_contract

In [ ]:
def category_summary(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    result = frame[column].value_counts(dropna=False).rename_axis(column).reset_index(name="n")
    result["share_percent"] = (100 * result["n"] / len(frame)).round(1)
    return result

def numeric_summary(frame: pd.DataFrame, group: str, value: str) -> pd.DataFrame:
    return (frame.groupby(group, dropna=False)[value]
            .agg(n="size", mean="mean", median="median")
            .round(1).reset_index())

print("Helpers loaded. Choose a helper only after you have stated what your question asks.")

## 1. Week 1 data contract

Complete this before you calculate anything.

- **What does one row represent?** _replace_
- **What data do you have?** _replace_
- **Population you would like to understand:** _replace_
- **Sampling frame or list that could have produced this data:** _replace_
- **Why the supplied sample may differ from the population:** _replace_

Choose two columns from data_contract and explain why storage type is not enough to tell you their meaning.

> _Write 2-3 sentences here._

## 2. Turn your vague brief into a question

Write a question that this **one table** can answer descriptively.

- **Vague brief in one sentence:** _replace_
- **Precise analytical question:** _replace_
- **Unit of observation:** _replace_
- **Population and inclusion rule:** _replace_
- **Variables and measurement types:** _replace_
- **Operational definition:** _replace_
- **Statistic or share you will report:** _replace_
- **One claim your analysis cannot make:** _replace_

### Plan before code

In 3-4 bullets, explain which variables you will use, what one row means, which summary answers your question, and one way your result could mislead a reader.

## 3. Build a descriptive answer

Use only students_sample. Do **not** merge tables in this lab. A good answer has a visible denominator, a meaningful unit, and words that match the evidence.

In [ ]:
# TODO 1 - Build analysis_frame with the columns needed for your question.
# Keep student_id, even if you do not display it in your final evidence table.
# Then display .head() and .shape.

pass

In [ ]:
# TODO 2 - Write two checks that support the unit and scope you claimed.
# One check must show whether student_id is unique.
# The second should check categories, missing values, or sample size relevant to your question.
# Write a one-sentence interpretation immediately below this cell.

pass

In [ ]:
# TODO 3 - Build and display evidence_table.
# It must show relevant groups/categories, n, and either a percentage or a numerical summary.
# Use category_summary(...) or numeric_summary(...) only when it fits your question.

pass

**Evidence interpretation (3-4 sentences):** Report the central pattern with its denominator. Then state one interpretation that your table does **not** justify.

> _Write here._

In [ ]:
# TODO 4 - Make one chart that helps a reader inspect evidence_table.
# Use an honest title and axis labels. Include group sizes in the table or chart.
# Hint: a bar chart may work for category shares; a grouped point/bar chart may work for numerical summaries.

pass

**Chart interpretation (2-3 sentences):** Explain what the chart helps the reader see. Explain one thing it cannot prove.

> _Write here._

## 4. Challenge the measurement and sample

Choose one relevant limitation.

- **Proxy limitation:** Does a column stand in for something more complex? Where could it fail?
- **Sample limitation:** What would you need to know before generalising from this sample to the university?
- **Measurement limitation:** What important distinction does the column fail to measure?
- **Comparison limitation:** Could category size, spread, or an omitted variable change how someone reads your result?

**Your limitation (3-4 sentences):** _Write here._

**Responsible next step:** Name one additional data source, qualitative question, or check that would improve the decision. _Write here._

## 5. Peer check

Show a nearby student your fingerprint, question, analysis_frame.shape, evidence table, and chart. The reviewer asks:

1. Does the question match what one row means?
2. Is the denominator visible?
3. Which sentence in the conclusion sounds stronger than the evidence?

- **Reviewer initials:** _replace_
- **Their challenge:** _replace_
- **What I revised:** _replace_

## 6. Final memo

Write **150-220 words** for the person in your brief. Include your question, sample, unit, key number, cautious conclusion, one limitation, and one responsible next step.

Avoid words such as “proves,” “causes,” or “all students” unless your data and design justify them.

> _Write your memo here._

In [ ]:
# Optional word-count check. Copy your finished memo here only if useful.
final_memo = ""
if final_memo.strip():
    print("Memo word count:", len(final_memo.split()))
else:
    print("Write the memo in the markdown cell above.")
print("Submission fingerprint:", ASSIGNMENT_FINGERPRINT)

## Submission checklist and marking guide

Before submitting: **Runtime -> Restart session and run all**.

- [ ] My fingerprint is visible.
- [ ] I used only the Week 1 one-table data.
- [ ] My question names the unit, sample, and variables.
- [ ] My evidence table includes group size or denominator.
- [ ] My chart and written explanation match the evidence.
- [ ] I state a real limitation and responsible next step.
- [ ] I completed the peer check and final memo.

| Criterion | Weight |
| --- | ---: |
| Question, unit, and measurement choices | 30% |
| Reproducible descriptive analysis | 25% |
| Evidence table, chart, and explanation | 25% |
| Sample/proxy limitation and responsible conclusion | 15% |
| Peer check and complete submission | 5% |